# Clean codon_variants_Omi-32.csv generated from the Bloom Lab's dms-vep-pipeline-3 (see '../link_genos_barcodes/README.md' for details on our specific usage of the pipeline)
- bring in csv
- remove barcodes with incorrect length
- remove any antibody variants that have non-designed mutations (from the Omi32 combinatorial library in Tharp et al, 2026)
- for the remaining barcodes/variants, convert the variants to binary genotypes (where '0' = wild-type/germline amino acid and '1' = mature/Omi32 amino acid) 

In [1]:
import os
import pandas as pd

In [2]:
variants = pd.read_csv("../link_genos_barcodes/results/variants/codon_variants_Omi-32.csv")
barcode_count = []
for barc in variants["barcode"]:
    barcode_count.append(len(barc))
variants = variants.assign(barcode_length=barcode_count)
variants = variants[variants['barcode_length'] == 22]
variants = variants.reset_index()
variants['aa_substitutions'] = variants['aa_substitutions'].astype("string")
variants["aa_substitutions"] = variants["aa_substitutions"].fillna('WT')

In [3]:
# Here mutations are numbered by their position in the full IgG construct (by RNA transcript, which includes both heavy and light chain, signal peptides, and constant regions)
## To align these mutations to their positions reported in Tharp et al, 2026, subtract 19 from the first six mutations (heavy chain) in this list (excluding 'WT') and subtract 557 from the final seven mutations (light chain)
correct_variants_list = ['WT','Q20E','S50N','I70Y','S75G','Y78F','A107V','E558A','V560R','L561M','V586I','Y590F','S651T','E664D',' ']
acceptable_set = set(correct_variants_list)
f=0
j=0
results = {}
variants_curated=pd.DataFrame()
for value in variants['aa_substitutions']:
    substrings = value.split()
    for substring in substrings:
        if substring in acceptable_set:
            results[substring] = "Accepted"
        else:
            results[substring] = "Not Accepted"
            j=j+1
    if j < 1:
        variants_curated = variants_curated._append(variants.iloc[f], ignore_index=True)
    j=0
    f=f+1

In [ ]:
binary_genos = []
variant_aa_list = ['Q20E','S50N','I70Y','S75G','Y78F','A107V','E558A','V560R','L561M','V586I','Y590F','S651T','E664D']
for aa_var in variants_curated["aa_substitutions"]:
    ind_subs = aa_var.split()
    s= [0] * len(variant_aa_list)
    for i, var in enumerate(variant_aa_list):
        if var in ind_subs:
            s[i] = 1
    binary_genos.append(''.join(map(str,s)))
    
variants_curated = variants_curated.assign(binary_genotypes=binary_genos)
variants_curated['binary_genotypes'] = variants_curated['binary_genotypes']
variants_curated.to_csv('variants_barcodes_binary-genos.csv', index=False)
print('number of antibody variants with only designed mutations and correct barcode length:', len(variants_curated))  

number of antibody variants with only designed mutations and correct barcode length: 126684
